In [1]:
#Pawel Maczuga and Maciej Paszynski (2023)
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from matplotlib.animation import FuncAnimation
from functools import partial
# from google.colab import files
import time
import os
from typing import Tuple
from pathlib import Path
import wandb
import yaml

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

c:\Users\Kamila\anaconda3\envs\torch_env\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


device(type='cuda')

In [2]:
torch.cuda.empty_cache()

## Parameters

Skalowanie danych przepuszczalności do przedziału [0, 10].


In [3]:
def load_and_scale_kq(path):
    kq_layer = np.load(path)
    log_kq = np.log10(kq_layer + 1e-6)
    max_scale = 10.0
    l_min, l_max = log_kq.min(), log_kq.max()
    k_scaled = (log_kq - l_min) / (l_max - l_min) * (max_scale - 1.0) + 1.0
    scale = (1.0, max_scale)
    return k_scaled, scale

In [4]:
# https://www.geeksforgeeks.org/python/swish-activation-function-in-pytorch/
# Swish Function 
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

In [5]:
def get_activation_fun(name: str):
    if name == "tanh":
        return nn.Tanh()
    elif name == "sigmoid":
        return nn.Sigmoid()
    elif name =="swish":
        return Swish()
    else:
        raise ValueError(f"Nieznana funkcja aktywacji: {name}")

In [6]:
def get_optimizer(model_params, name: str, learning_rate: float, max_iter: int):
    if name == "Adam":
        return torch.optim.Adam(model_params, lr=learning_rate)
    elif name == "lbfgs":
        return torch.optim.LBFGS(model_params, lr=learning_rate, max_iter=max_iter, history_size=50, line_search_fn="strong_wolfe")
    else:
        raise ValueError(f"Nieznany optymalizer: {name}")

Pobieramy dane - te same, które wykorzystaliśmy w rozwiązaniu referencyjnym. Te dane to macierz o wymiarach 60 x 220. Rzeczywiste wymiary w metrach to ok. 366 x 671.

In [7]:
DATA_PATH = 'C:\PINN_mgr\FEniCS\solver-validation-newton-method\data\spe_model2_layer50.npy'
kq_data_matrix, scale = load_and_scale_kq(DATA_PATH)
ACTUAL_NY, ACTUAL_NX = kq_data_matrix.shape
# print(ACTUAL_NX, ACTUAL_NY)
REAL_NX = ACTUAL_NX
REAL_NY = ACTUAL_NY

FT_TO_M = 0.3048
REAL_LX = REAL_NX * 20.0 * FT_TO_M
# print(REAL_LX)
REAL_LY = REAL_NY * 10.0 * FT_TO_M
# print(REAL_LY)

In [8]:
TOTAL_TIME = 1000.                
N_POINTS_X = REAL_NX  
N_POINTS_Y = REAL_NY  

x_domain = [0.0, REAL_LX]
y_domain = [0.0, REAL_LY]
t_domain = [0.0, TOTAL_TIME]

## Initial condition

In [9]:
def initial_condition(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    mid_x = REAL_LX / 2.0
    mid_y = REAL_LY / 2.0

    r = torch.sqrt((x - mid_x)**2 + (y - mid_y)**2)
    return (r <= 30.0).float()

## PINN


In [10]:
class PINN(nn.Module):
    """Simple neural network accepting two features as input and returning a single output

    In the context of PINNs, the neural network is used as universal function approximator
    to approximate the solution of the differential equation
    """
    def __init__(self, num_hidden: int, dim_hidden: int, act=nn.Tanh()):

        super().__init__()

        self.layer_in = nn.Linear(3, dim_hidden)
        self.layer_out = nn.Linear(dim_hidden, 1)

        num_middle = num_hidden - 1
        self.middle_layers = nn.ModuleList(
            [nn.Linear(dim_hidden, dim_hidden) for _ in range(num_middle)]
        )
        self.act = act

    def forward(self, x, y, t):

        #normalizacja 
        x_norm = x / REAL_LX
        y_norm = y / REAL_LY
        t_norm = t / TOTAL_TIME

        x_stack = torch.cat([x_norm, y_norm, t_norm], dim=1)
        out = self.act(self.layer_in(x_stack))
        for layer in self.middle_layers:
            out = self.act(layer(out))
        logits = self.layer_out(out)

        return logits

    def device(self):
        return next(self.parameters()).device


def f(pinn: PINN, x: torch.Tensor, y: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """Compute the value of the approximate solution from the NN model"""
    return pinn(x, y, t)


def df(output: torch.Tensor, input: torch.Tensor, order: int = 1) -> torch.Tensor:
    """Compute neural network derivative with respect to input features using PyTorch autograd engine"""
    df_value = output
    for _ in range(order):
        df_value = torch.autograd.grad(
            df_value,
            input,
            grad_outputs=torch.ones_like(input),
            create_graph=True,
            retain_graph=True,
        )[0]

    return df_value


def dfdt(pinn: PINN, x: torch.Tensor, y: torch.Tensor, t: torch.Tensor, order: int = 1):
    f_value = f(pinn, x, y, t)
    return df(f_value, t, order=order)

def dfdx(pinn: PINN, x: torch.Tensor, y: torch.Tensor, t: torch.Tensor, order: int = 1):
    f_value = f(pinn, x, y, t)
    return df(f_value, x, order=order)

def dfdy(pinn: PINN, x: torch.Tensor, y: torch.Tensor, t: torch.Tensor, order: int = 1):
    f_value = f(pinn, x, y, t)
    return df(f_value, y, order=order)

## Loss function

In [11]:
def get_boundary_points(x_domain, y_domain, t_domain, device="cpu", requires_grad=True, n_samples=1000, n_points_t=20, n_time_points_perbin=10):
    """
         .+------+
       .' |    .'|
      +---+--+'  |
      |   |  |   |
    y |  ,+--+---+
      |.'    | .' t
      +------+'
         x
    """
    
    x = torch.rand(n_samples, 1, device=device) * (x_domain[1] - x_domain[0]) + x_domain[0]
    y = torch.rand(n_samples, 1, device=device) * (y_domain[1] - y_domain[0]) + y_domain[0]

    t_pool_list = []
    t_edges = np.linspace(t_domain[0], t_domain[1], n_points_t + 1)
    
    for i in range(n_points_t):
        t_min, t_max = t_edges[i], t_edges[i+1]
        t_bin = torch.rand(n_time_points_perbin, 1, device=device) * (t_max - t_min) + t_min
        t_pool_list.append(t_bin)
        
    t_pool = torch.cat(t_pool_list, dim=0)
    
    total_time_points = t_pool.shape[0]
    random_indices = torch.randint(0, total_time_points, (n_samples,), device=device)
    t = t_pool[random_indices]

    x_left = torch.zeros_like(x) + x_domain[0]
    x_right = torch.zeros_like(x) + x_domain[1]

    y_bottom = torch.zeros_like(y) + y_domain[0]
    y_top = torch.zeros_like(y) + y_domain[1]

    if requires_grad:
        for tensor in [x_left, x_right, y, y_bottom, y_top, x, t]:
            tensor.requires_grad_(True)

    return [
        (x_left, y, t),
        (x_right, y, t),
        (x, y_bottom, t),
        (x, y_top, t),
    ]

In [12]:
def get_initial_points(x_domain, y_domain, t_domain, device="cpu", requires_grad=True, n_samples=200):
    x = torch.rand(n_samples, 1, device=device) * (x_domain[1] - x_domain[0]) + x_domain[0]
    y = torch.rand(n_samples, 1, device=device) * (y_domain[1] - y_domain[0]) + y_domain[0]
    t = torch.zeros_like(x) + t_domain[0]

    if requires_grad:
        x.requires_grad = True
        y.requires_grad = True
        t.requires_grad = True

    return x, y, t

In [13]:
def get_interior_points(x_domain, y_domain, t_domain, device=torch.device("cpu"), requires_grad=True, n_samples=1000, n_points_t=20, n_time_points_perbin=10):
    x = torch.rand(n_samples, 1, device=device) * (x_domain[1] - x_domain[0]) + x_domain[0]
    y = torch.rand(n_samples, 1, device=device) * (y_domain[1] - y_domain[0]) + y_domain[0]
    # t = torch.rand(n_samples, 1, device=device) * (t_domain[1] - t_domain[0]) + t_domain[0]

    t_pool_list = []
    t_edges = np.linspace(t_domain[0], t_domain[1], n_points_t + 1)
    
    for i in range(n_points_t):
        t_min, t_max = t_edges[i], t_edges[i+1]
        t_bin = torch.rand(n_time_points_perbin, 1, device=device) * (t_max - t_min) + t_min
        t_pool_list.append(t_bin)
        
    t_pool = torch.cat(t_pool_list, dim=0) 
    
    total_time_points = t_pool.shape[0]
    random_indices = torch.randint(0, total_time_points, (n_samples,), device=device)
    
    t = t_pool[random_indices]
    

    if requires_grad:
        x.requires_grad = True
        y.requires_grad = True
        t.requires_grad = True

    return x, y, t

In [14]:
KQ_TENSOR = torch.tensor(kq_data_matrix, dtype=torch.float32).to(device)

In [15]:
def Kq(x, y):
    x_norm = x / REAL_LX
    y_norm = y / REAL_LY
    
    idx_x = (x_norm * (REAL_NX - 1)).long()
    idx_y = (y_norm * (REAL_NY - 1)).long()

    idx_x = torch.clamp(idx_x, 0, REAL_NX - 1)
    idx_y = torch.clamp(idx_y, 0, REAL_NY - 1)

    return KQ_TENSOR[idx_y, idx_x]

def K(x, y, u):
    return Kq(x, y) * torch.exp(10.0 * u)

def source_term(name, x, y):
    return 0.0
    # if name == "h=0":
        # return 0.0
    # else:
    #     x_n = x / REAL_LX
    #     y_n = y / REAL_LY
    #     return 1.0 + torch.sin(2 * np.pi * x_n) * torch.sin(2 * np.pi * y_n)

In [16]:
class Loss:
    def __init__(
        self,
        x_domain: Tuple[float, float],
        y_domain: Tuple[float, float],
        t_domain: Tuple[float, float],
        N_POINTS_T: int,
        initial_condition: Callable,
        weight_r: float = 1.0,
        weight_b: float = 1.0,
        weight_i: float = 1.0,
        verbose: bool = False,
        n_interior: int = 1000,
        n_boundary: int = 1000,
        n_initial: int = 1000,
        fixed_points=None
    ):
        self.x_domain = x_domain
        self.y_domain = y_domain
        self.t_domain = t_domain
        self.N_POINTS_T = N_POINTS_T
        self.initial_condition = initial_condition
        self.weight_r = weight_r
        self.weight_b = weight_b
        self.weight_i = weight_i
        self.n_interior = n_interior
        self.n_boundary = n_boundary
        self.n_initial = n_initial
        self.fixed_points = fixed_points

    def residual_loss(self, pinn: PINN):
        if self.fixed_points is not None:
            x, y, t = self.fixed_points["interior"]
        else:
            x, y, t = get_interior_points(self.x_domain, self.y_domain, self.t_domain, pinn.device(), n_samples=self.n_interior, n_points_t=self.N_POINTS_T, n_time_points_perbin=10)

        u = f(pinn, x, y, t)
        
        du_dx = dfdx(pinn, x, y, t)
        du_dy = dfdy(pinn, x, y, t)
        
        K_val = K(x, y, u)

        flux_x = K_val * du_dx
        flux_y = K_val * du_dy

        d_flux_x_dx = df(flux_x, x)
        d_flux_y_dy = df(flux_y, y)
        div_flux = d_flux_x_dx + d_flux_y_dy

        u_t = dfdt(pinn, x, y, t)
        source = source_term("h=0", x, y)
        res = u_t - div_flux - source

        return res.pow(2).mean()

    def initial_loss(self, pinn: PINN):
        if self.fixed_points is not None:
            x, y, t = self.fixed_points["initial"]
        else:
            x, y, t = get_initial_points(self.x_domain, self.y_domain, self.t_domain, pinn.device(), n_samples=self.n_initial)

        pinn_init = self.initial_condition(x, y)
        loss = f(pinn, x, y, t) - pinn_init
        return loss.pow(2).mean()

    def boundary_loss(self, pinn: PINN):
        if self.fixed_points is not None:
            left, right, bottom, top = self.fixed_points["boundary"]
        else:
            left, right, bottom, top = get_boundary_points(self.x_domain, self.y_domain, self.t_domain, pinn.device(), n_samples=self.n_boundary, n_points_t=self.N_POINTS_T, n_time_points_perbin=10)

        u_left   = f(pinn, *left)
        u_right  = f(pinn, *right)
        u_bottom = f(pinn, *bottom)
        u_top    = f(pinn, *top)

        loss = u_left.pow(2).mean() + \
                u_right.pow(2).mean() + \
                u_bottom.pow(2).mean() + \
                u_top.pow(2).mean()

        return loss

    def verbose(self, pinn: PINN):
        """
        Returns all parts of the loss function

        Not used during training! Only for checking the results later.
        """
        residual_loss = self.residual_loss(pinn)
        initial_loss = self.initial_loss(pinn)
        boundary_loss = self.boundary_loss(pinn)

        final_loss = \
            self.weight_r * residual_loss + \
            self.weight_i * initial_loss + \
            self.weight_b * boundary_loss

        return final_loss, residual_loss, initial_loss, boundary_loss

    def __call__(self, pinn: PINN):
        """
        Allows you to use instance of this class as if it was a function:

        ```
            >>> loss = Loss(*some_args)
            >>> calculated_loss = loss(pinn)
        ```
        """
        return self.verbose(pinn)[0]

## Plottig functions and metrics

In [17]:
def calculate_metrics(model, t_value, u_fenics_tensor):
    model.eval()
    with torch.no_grad():
        x_linspace = torch.linspace(0, REAL_LX, REAL_NX, device=device)
        y_linspace = torch.linspace(0, REAL_LY, REAL_NY, device=device) 
        X, Y = torch.meshgrid(x_linspace, y_linspace, indexing="ij")
        
        X_flat = X.reshape(-1, 1)
        Y_flat = Y.reshape(-1, 1)
        T_flat = torch.full_like(X_flat, t_value)

        u_pred_flat = model(X_flat, Y_flat, T_flat)
        u_pred = u_pred_flat.reshape(REAL_NX, REAL_NY)

        max_u = torch.max(u_pred).item()
        mean_u = torch.mean(u_pred).item()
        energy = torch.mean(u_pred**2).item() 

        diff = u_pred - u_fenics_tensor
        l2_error = torch.norm(diff) / torch.norm(u_fenics_tensor)

        return {
            "max_u": max_u,
            "mean_u": mean_u,
            "energy": energy,
            "l2_error": l2_error.item(),
            "u_map": u_pred.cpu().numpy() 
        }

In [18]:
def get_pinn_map(model, t_value, name):
    model.eval()
    with torch.no_grad():
        x_linspace = torch.linspace(x_domain[0], x_domain[1], N_POINTS_X, device=device)
        y_linspace = torch.linspace(y_domain[0], y_domain[1], N_POINTS_Y, device=device)
        X, Y = torch.meshgrid(x_linspace, y_linspace, indexing="ij")
        
        T = torch.full_like(X, t_value)
        u_pred = model(X.reshape(-1, 1), Y.reshape(-1, 1), T.reshape(-1, 1))
        u_map = u_pred.reshape(N_POINTS_X, N_POINTS_Y).cpu().numpy()
        
        save_dir = Path("results/maps_to_compare")
        save_dir.mkdir(parents=True, exist_ok=True)

        file_path = save_dir / name

        data_to_save = {
            "u_pinn": u_map,
            "t": t_value,
            "x_grid": X.cpu().numpy(),
            "y_grid": Y.cpu().numpy()
        }
        
        np.save(file_path, data_to_save)

        return u_map

In [19]:
def plot_color(z, title, cmap='jet'):
    fig, ax = plt.subplots(figsize=(10, 14), dpi=100)
    
    settings = {'origin': 'lower', 'extent': [0, REAL_LX, 0, REAL_LY], 'aspect': 'equal'}
    
    im = ax.imshow(z.T, cmap=cmap, **settings)
    
    ax.set_title(title, fontsize=20)
    ax.set_xlabel("x [m]", fontsize=20)
    ax.set_ylabel("y [m]", fontsize=20)
    
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Ciśnienie u", fontsize=20) 
    cbar.ax.tick_params(labelsize=20, rotation=35)          
    
    ax.tick_params(labelsize=20, rotation=35)
    
    # plt.tight_layout()
    return fig

In [20]:
def plot_pinn_vs_fenics(u_pinn, u_fenics, t_value):
    error_map = np.abs(u_pinn - u_fenics)
    l2_error = np.linalg.norm(u_fenics - u_pinn) / np.linalg.norm(u_fenics)

    fig, axes = plt.subplots(1, 3, figsize=(24, 8), dpi=100)
    settings = {'origin': 'lower', 'extent': [0, REAL_LX, 0, REAL_LY], 'aspect': 'equal'}

    #FEniCS
    im0 = axes[0].imshow(u_fenics.T, cmap='jet', **settings)
    axes[0].set_title(f"FEniCS (Referencja) t={t_value}", fontsize=20)
    cbar0 = fig.colorbar(im0, ax=axes[0])
    cbar0.set_label("Ciśnienie u", fontsize=20) 
    cbar0.ax.tick_params(labelsize=20, rotation=35)          

    #PINN
    im1 = axes[1].imshow(u_pinn.T, cmap='jet', **settings)
    axes[1].set_title(f"PINN (Model) t={t_value}", fontsize=20)
    cbar1 = fig.colorbar(im1, ax=axes[1])
    cbar1.set_label("Ciśnienie u", fontsize=20)
    cbar1.ax.tick_params(labelsize=20, rotation=35)

    #Error
    im2 = axes[2].imshow(error_map.T, cmap='hot', **settings)
    axes[2].set_title(f"Błąd bezwględny (L2={l2_error:.3f})", fontsize=20)
    cbar2 = fig.colorbar(im2, ax=axes[2])
    cbar2.set_label("Różnica ciśnienia u", fontsize=20)
    cbar2.ax.tick_params(labelsize=20, rotation=35)

    for ax in axes:
        ax.set_xlabel("x [m]", fontsize=20)
        ax.set_ylabel("y [m]", fontsize=20)
        ax.tick_params(axis='both', which='major', labelsize=20, rotation=0) 

    # plt.tight_layout()
    return fig

In [21]:
file_path_u_f_final = r'C:\PINN_mgr\FEniCS\solver-validation-newton-method\results\metrics\fenics_for_pinn_T1000_dt1_u_coeff10_h_0.npy'

fenics_data = np.load(file_path_u_f_final, allow_pickle=True).item()
u_f_final = fenics_data['u_fenics'].T

## Train function

In [22]:
saved_models = Path(r"C:\\PINN_mgr\\maczuga_pinn\\PINN\\models_T1000_h0_AGAIN")

In [23]:
def train_with_wandb_adam(params, run):

    run_name = f"lr{params['layers']}_neur{params['neurons_per_layer']}_nep{params['num_epochs']}_act_{params['activation_function']}_opt{params['optimizer']}_lr{params['learning_rate']}_inte_p{params['interior_points']}_bound_p{params['boundary_points']}_init_p{params['initial_points']}"
    run.name = run_name
    
    model = PINN(
        num_hidden=params['layers'],
        dim_hidden=params['neurons_per_layer'],
        act=get_activation_fun(params['activation_function'])
    ).to(device)

    interior_init = get_interior_points(x_domain, y_domain, t_domain, device=device, n_samples=params['interior_points'], n_points_t=params['N_POINTS_T'], n_time_points_perbin=10)
    boundary_init = get_boundary_points(x_domain, y_domain, t_domain, device=device, n_samples=params['boundary_points'], n_points_t=params['N_POINTS_T'], n_time_points_perbin=10)
    initial_init = get_initial_points(x_domain, y_domain, t_domain, device=device, n_samples=params['initial_points'])

    t_interior_fixed = interior_init[2]
    t_initial_fixed = initial_init[2]
    t_boundary_fixed = [pt[2] for pt in boundary_init]

    # interior_pts = get_interior_points(
    # x_domain, y_domain, t_domain,
    # device=device,
    # n_samples=params['interior_points'],
    # n_points_t=params['N_POINTS_T'],
    # n_time_points_perbin=10
    # )

    # boundary_pts = get_boundary_points(
    #     x_domain, y_domain, t_domain,
    #     device=device,
    #     n_samples=params['boundary_points'],
    #     n_points_t=params['N_POINTS_T'],
    #     n_time_points_perbin=10
    # )

    # initial_pts = get_initial_points(
    #     x_domain, y_domain, t_domain,
    #     device=device,
    #     n_samples=params['initial_points']
    # )

    # fixed_points = {
    #     "interior": interior_pts,
    #     "boundary": boundary_pts,
    #     "initial": initial_pts
    # }

    loss_fn = Loss(
        x_domain=x_domain, y_domain=y_domain, t_domain=t_domain,
        N_POINTS_T=params['N_POINTS_T'],
        initial_condition=initial_condition,
        weight_r=params['weight_residual'],
        weight_b=params['weight_boundary'],
        weight_i=params['weight_initial'],
        n_interior=params['interior_points'],
        n_boundary=params['boundary_points'],
        n_initial=params['initial_points'],
        fixed_points={
            "interior": interior_init,
            "boundary": boundary_init,
            "initial": initial_init
        }
        # ,
        # fixed_points=fixed_points
    )

    optimizer = get_optimizer(model.parameters(), "Adam", params['learning_rate'], 0)
    
    u_f_tensor_final = torch.tensor(u_f_final, device=device, dtype=torch.float32)


    percentages = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.4, 0.6, 0.8, 1.0]
    log_intervals = [int(p * params['num_epochs']) for p in percentages]
    
    model.eval()
    with torch.no_grad():
        metrics_final = calculate_metrics(model, TOTAL_TIME, u_f_tensor_final)

        fig_final = plot_color(metrics_final["u_map"], "Epoka 0 (Inicjalizacja)")

        wandb.log({
            "evolution/1.0_final": wandb.Image(fig_final)
        }, step=0)

        plt.close(fig_final); 

    for epoch in range(params['num_epochs']):
        model.train() 

        x_int, y_int, _ = get_interior_points(x_domain, y_domain, t_domain, device=device, n_samples=params['interior_points'])
        x_init, y_init, _ = get_initial_points(x_domain, y_domain, t_domain, device=device, n_samples=params['initial_points'])
        b_pts = get_boundary_points(x_domain, y_domain, t_domain, device=device, n_samples=params['boundary_points'])

        # Wstrzykujemy nową przestrzeń z zachowaniem starego czasu
        loss_fn.fixed_points = {
            "interior": (x_int, y_int, t_interior_fixed),
            "initial": (x_init, y_init, t_initial_fixed),
            "boundary": [
                (b_pts[0][0], b_pts[0][1], t_boundary_fixed[0]), # x_left, y, t_fixed
                (b_pts[1][0], b_pts[1][1], t_boundary_fixed[1]), # x_right, y, t_fixed
                (b_pts[2][0], b_pts[2][1], t_boundary_fixed[2]), # x, y_bottom, t_fixed
                (b_pts[3][0], b_pts[3][1], t_boundary_fixed[3]), # x, y_top, t_fixed
            ]
        }

        total_l, res_l, init_l, bnd_l = loss_fn.verbose(model)
        optimizer.zero_grad()
        total_l.backward()
        optimizer.step()

        if epoch % 10 == 0 or epoch == params['num_epochs'] - 1 or (epoch + 1) in log_intervals:
            metrics_final = calculate_metrics(model, TOTAL_TIME, u_f_tensor_final)

            logs = {}

            if epoch % 10 == 0 or epoch == params['num_epochs'] - 1:
                logs.update({
                    "losses/total": total_l.item(),
                    "losses/residual": res_l.item(),
                    "losses/initial": init_l.item(),
                    "losses/boundary": bnd_l.item(),
                    
                    "t1/l2_error": metrics_final["l2_error"],
                    "t1/max_u": metrics_final["max_u"],
                    "t1/energy": metrics_final["energy"],
                    "t1/mean_u": metrics_final["mean_u"]
                })

            if (epoch + 1) in log_intervals:
                fig_final = plot_color(metrics_final["u_map"], f"Epoka {epoch+1}")
                logs["evolution/1.0_final"] = wandb.Image(fig_final)

            if logs:
                wandb.log(logs, step=epoch)
                
            if (epoch + 1) in log_intervals:
                plt.close(fig_final)

    u_p_final = get_pinn_map(model, t_value=TOTAL_TIME, name=f"pinn_map_final_{run_name}.npy")
    fig_comp_fin = plot_pinn_vs_fenics(metrics_final["u_map"], u_f_final, TOTAL_TIME)
    
    wandb.log({"comparison/final_comparison_plot": wandb.Image(fig_comp_fin)
                })
    
    plt.close(fig_comp_fin)

    model_filename = f"{run_name}.pth"
    full_path = saved_models / model_filename 

    torch.save(model.state_dict(), str(full_path)) 

    model_artifact = wandb.Artifact(
        name=f"model_{run.id}", 
        type='model'
    )
    model_artifact.add_file(str(full_path)) 
    run.log_artifact(model_artifact)

In [24]:
# def train_with_wandb_adam(params, run):

#     run_name = f"lr{params['layers']}_neur{params['neurons_per_layer']}_nep{params['num_epochs']}_act_{params['activation_function']}_opt{params['optimizer']}_lr{params['learning_rate']}_inte_p{params['interior_points']}_bound_p{params['boundary_points']}_init_p{params['initial_points']}"
#     run.name = run_name
    
#     model = PINN(
#         num_hidden=params['layers'],
#         dim_hidden=params['neurons_per_layer'],
#         act=get_activation_fun(params['activation_function'])
#     ).to(device)

#     # interior_pts = get_interior_points(
#     # x_domain, y_domain, t_domain,
#     # device=device,
#     # n_samples=params['interior_points'],
#     # n_points_t=params['N_POINTS_T'],
#     # n_time_points_perbin=10
#     # )

#     # boundary_pts = get_boundary_points(
#     #     x_domain, y_domain, t_domain,
#     #     device=device,
#     #     n_samples=params['boundary_points'],
#     #     n_points_t=params['N_POINTS_T'],
#     #     n_time_points_perbin=10
#     # )

#     # initial_pts = get_initial_points(
#     #     x_domain, y_domain, t_domain,
#     #     device=device,
#     #     n_samples=params['initial_points']
#     # )

#     # fixed_points = {
#     #     "interior": interior_pts,
#     #     "boundary": boundary_pts,
#     #     "initial": initial_pts
#     # }

#     loss_fn = Loss(
#         x_domain=x_domain, y_domain=y_domain, t_domain=t_domain,
#         N_POINTS_T=params['N_POINTS_T'],
#         initial_condition=initial_condition,
#         weight_r=params['weight_residual'],
#         weight_b=params['weight_boundary'],
#         weight_i=params['weight_initial'],
#         n_interior=params['interior_points'],
#         n_boundary=params['boundary_points'],
#         n_initial=params['initial_points']
#         # ,
#         # fixed_points=fixed_points
#     )

#     optimizer = get_optimizer(model.parameters(), "Adam", params['learning_rate'], 0)
    
#     u_f_tensor_final = torch.tensor(u_f_final, device=device, dtype=torch.float32)


#     percentages = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.4, 0.6, 0.8, 1.0]
#     log_intervals = [int(p * params['num_epochs']) for p in percentages]
    
#     model.eval()
#     with torch.no_grad():
#         metrics_final = calculate_metrics(model, TOTAL_TIME, u_f_tensor_final)

#         fig_final = plot_color(metrics_final["u_map"], "Epoka 0 (Inicjalizacja)")

#         wandb.log({
#             "evolution/1.0_final": wandb.Image(fig_final)
#         }, step=0)

#         plt.close(fig_final); 

#     for epoch in range(params['num_epochs']):
#         model.train() 

#         total_l, res_l, init_l, bnd_l = loss_fn.verbose(model)
#         optimizer.zero_grad()
#         total_l.backward()
#         optimizer.step()

#         if epoch % 10 == 0 or epoch == params['num_epochs'] - 1 or (epoch + 1) in log_intervals:
#             metrics_final = calculate_metrics(model, TOTAL_TIME, u_f_tensor_final)

#             logs = {}

#             if epoch % 10 == 0 or epoch == params['num_epochs'] - 1:
#                 logs.update({
#                     "losses/total": total_l.item(),
#                     "losses/residual": res_l.item(),
#                     "losses/initial": init_l.item(),
#                     "losses/boundary": bnd_l.item(),
                    
#                     "t1/l2_error": metrics_final["l2_error"],
#                     "t1/max_u": metrics_final["max_u"],
#                     "t1/energy": metrics_final["energy"],
#                     "t1/mean_u": metrics_final["mean_u"]
#                 })

#             if (epoch + 1) in log_intervals:
#                 fig_final = plot_color(metrics_final["u_map"], f"Epoka {epoch+1}")
#                 logs["evolution/1.0_final"] = wandb.Image(fig_final)

#             if logs:
#                 wandb.log(logs, step=epoch)
                
#             if (epoch + 1) in log_intervals:
#                 plt.close(fig_final)

#     u_p_final = get_pinn_map(model, t_value=TOTAL_TIME, name=f"pinn_map_final_{run_name}.npy")
#     fig_comp_fin = plot_pinn_vs_fenics(metrics_final["u_map"], u_f_final, TOTAL_TIME)
    
#     wandb.log({"comparison/final_comparison_plot": wandb.Image(fig_comp_fin)
#                 })
    
#     plt.close(fig_comp_fin)

#     model_filename = f"{run_name}.pth"
#     full_path = saved_models / model_filename 

#     torch.save(model.state_dict(), str(full_path)) 

#     model_artifact = wandb.Artifact(
#         name=f"model_{run.id}", 
#         type='model'
#     )
#     model_artifact.add_file(str(full_path)) 
#     run.log_artifact(model_artifact)

In [25]:
def train_with_wandb_lbfgs(params, run):


    run_name = f"lr{params['layers']}_neur{params['neurons_per_layer']}_nep{params['num_epochs']}_act_{params['activation_function']}_opt{params['optimizer']}_lr{params['learning_rate']}_inte_p{params['interior_points']}_bound_p{params['boundary_points']}_init_p{params['initial_points']}_maxiter{params['max_iter']}"
    run.name = run_name
    
    model = PINN(
        num_hidden=params['layers'],
        dim_hidden=params['neurons_per_layer'],
        act=get_activation_fun(params['activation_function'])
    ).to(device)

    interior_pts = get_interior_points(
    x_domain, y_domain, t_domain,
    device=device,
    n_samples=params['interior_points'],
    n_points_t=params['N_POINTS_T'],
    n_time_points_perbin=10
    )

    boundary_pts = get_boundary_points(
        x_domain, y_domain, t_domain,
        device=device,
        n_samples=params['boundary_points'],
        n_points_t=params['N_POINTS_T'],
        n_time_points_perbin=10
    )

    initial_pts = get_initial_points(
        x_domain, y_domain, t_domain,
        device=device,
        n_samples=params['initial_points']
    )

    fixed_points = {
        "interior": interior_pts,
        "boundary": boundary_pts,
        "initial": initial_pts
    }

    loss_fn = Loss(
        x_domain=x_domain, y_domain=y_domain, t_domain=t_domain,
        N_POINTS_T=params['N_POINTS_T'],
        initial_condition=initial_condition,
        weight_r=params['weight_residual'],
        weight_b=params['weight_boundary'],
        weight_i=params['weight_initial'],
        n_interior=params['interior_points'],
        n_boundary=params['boundary_points'],
        n_initial=params['initial_points'],
        fixed_points=fixed_points
    )

    optimizer = get_optimizer(model.parameters(),"lbfgs", params['learning_rate'], params['max_iter'])
    
    u_f_tensor_final = torch.tensor(u_f_final, device=device, dtype=torch.float32)


    percentages = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.4, 0.6, 0.8, 1.0]
    log_intervals = [int(p * params['num_epochs']) for p in percentages]
    
    model.eval()
    with torch.no_grad():
        metrics_final = calculate_metrics(model, TOTAL_TIME, u_f_tensor_final)

        fig_final = plot_color(metrics_final["u_map"], "Epoka 0 (Inicjalizacja)")

        wandb.log({
            "evolution/1.0_final": wandb.Image(fig_final)
        }, step=0)

        plt.close(fig_final)

    

    for epoch in range(params['num_epochs']):
        model.train() 

        # https://discuss.pytorch.org/t/try-to-understand-how-to-use-optim-lbfgs/195602/5
        def closure():
            optimizer.zero_grad()
            total_l, _, _, _ = loss_fn.verbose(model)
            total_l.backward()
            return total_l

        optimizer.step(closure)
        # with torch.no_grad():
        total_l, res_l, init_l, bnd_l = loss_fn.verbose(model)
        
        if epoch % 10 == 0 or epoch == params['num_epochs'] - 1 or (epoch + 1) in log_intervals:
            metrics_final = calculate_metrics(model, TOTAL_TIME, u_f_tensor_final)

            logs = {}

            if epoch % 10 == 0 or epoch == params['num_epochs'] - 1:
                logs.update({
                    "losses/total": total_l.item(),
                    "losses/residual": res_l.item(),
                    "losses/initial": init_l.item(),
                    "losses/boundary": bnd_l.item(),
                    
                    "t1/l2_error": metrics_final["l2_error"],
                    "t1/max_u": metrics_final["max_u"],
                    "t1/energy": metrics_final["energy"],
                    "t1/mean_u": metrics_final["mean_u"]
                })

            if (epoch + 1) in log_intervals:
                fig_final = plot_color(metrics_final["u_map"], f"Epoka {epoch+1}")
                logs["evolution/1.0_final"] = wandb.Image(fig_final)

            if logs:
                wandb.log(logs, step=epoch)
                
            if (epoch + 1) in log_intervals:
                plt.close(fig_final)

            

    u_p_final = get_pinn_map(model, t_value=TOTAL_TIME, name=f"pinn_map_final_{run_name}.npy")
    fig_comp_fin = plot_pinn_vs_fenics(metrics_final["u_map"], u_f_final, TOTAL_TIME)
    
    wandb.log({"comparison/final_comparison_plot": wandb.Image(fig_comp_fin),
                })
    
    plt.close(fig_comp_fin)

    model_filename = f"{run_name}.pth"
    full_path = saved_models / model_filename 

    torch.save(model.state_dict(), str(full_path)) 

    model_artifact = wandb.Artifact(
        name=f"model_{run.id}", 
        type='model'
    )
    model_artifact.add_file(str(full_path)) 
    run.log_artifact(model_artifact)

In [26]:
def train_with_wandb():
    with wandb.init() as run:
        config = run.config
        params = wandb.config.set

        optimizer_name = params['optimizer']
        if optimizer_name == "lbfgs":
            train_with_wandb_lbfgs(params, run)
        else:
            train_with_wandb_adam(params, run)
       

## Train data with WanDB

In [ ]:
with open('C:\PINN_mgr\maczuga_pinn\sweeph0.yaml', 'r', encoding='utf-8') as plik:
    sweep_config = yaml.safe_load(plik)

sweep_id = wandb.sweep(sweep_config, project="VanillaPINN_T1000_h0_pointsfixedALL")

wandb.agent(sweep_id, function=train_with_wandb) #, count=1 --->liczba prób

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Kamila\_netrc.


Create sweep with ID: nif0szmu
Sweep URL: https://wandb.ai/kamila-cwikla8-mgr/VanillaPINN_T1000_h0_pointsfixedALL/sweeps/nif0szmu


wandb: Agent Starting Run: e6t2rclh with config:
wandb: 	set: {'N_POINTS_T': 20, 'activation_function': 'tanh', 'boundary_points': 15000, 'initial_points': 15000, 'interior_points': 5000, 'layers': 8, 'learning_rate': 0.5, 'max_iter': 20, 'neurons_per_layer': 32, 'num_epochs': 200, 'optimizer': 'lbfgs', 'weight_boundary': 1, 'weight_initial': 1, 'weight_residual': 1}
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Kamila\_netrc.
wandb: Currently logged in as: kamila-cwikla8 (kamila-cwikla8-mgr) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


c:\Users\Kamila\anaconda3\envs\torch_env\lib\site-packages\torch\autograd\graph.py:824: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:181.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
